In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

block_size = 64
batch_size = 128
max_iters = 3000
eval_iters = 100
learning_rate = 3e-4
n_embd = 384
n_head = 8
n_layer = 8
dropout = 0.2




cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
with open("/content/drive/MyDrive/datasets/wizard_of_oz.txt", "r", encoding="utf-8") as f:
    text = f.read()

    chars = sorted(set(text))

    print(len(chars))





80


In [ ]:
string_to_int = {ch:i for i,ch in enumerate(chars)}
int_to_string = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype = torch.long)
print(data[:100])

tensor([ 0,  0,  0,  0, 28, 39, 42, 39, 44, 32, 49,  1, 25, 38, 28,  1, 44, 32,
        29,  1, 47, 33, 50, 25, 42, 28,  1, 33, 38,  1, 39, 50,  0,  0, 26, 49,
         0,  0, 36, 11,  1, 30, 42, 25, 38, 35,  1, 26, 25, 45, 37,  0,  0, 25,
        45, 44, 32, 39, 42,  1, 39, 30,  1, 44, 32, 29,  1, 47, 33, 50, 25, 42,
        28,  1, 39, 30,  1, 39, 50,  9,  1, 44, 32, 29,  1, 36, 25, 38, 28,  1,
        39, 30,  1, 39, 50,  9,  1, 39, 50, 37])


In [ ]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data) - block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  x,y = x.to(device), y.to(device)
  return x,y

x,y = get_batch('train')
print('inputs')
print(x)
print('targets')
print(y)





inputs
tensor([[ 1, 73, 61,  ..., 76, 62, 73],
        [54, 60, 58,  ...,  1, 76, 54],
        [73, 68,  1,  ...,  1, 76, 58],
        ...,
        [58, 54, 71,  ..., 59,  0, 73],
        [61, 68, 74,  ..., 62, 72, 73],
        [54, 73, 56,  ..., 72, 54, 75]], device='cuda:0')
targets
tensor([[73, 61, 62,  ..., 62, 73, 61],
        [60, 58,  1,  ..., 76, 54, 72],
        [68,  1, 72,  ..., 76, 58,  1],
        ...,
        [54, 71, 72,  ...,  0, 73, 71],
        [68, 74, 60,  ..., 72, 73, 54],
        [73, 56, 61,  ..., 54, 75, 58]], device='cuda:0')


In [ ]:

@ torch.no_grad()
def estimate_loss():
  out = {}
  model.eval()
  for split in ['train','val']:
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
      X,Y = get_batch(split)
      logits,loss = model(X,Y)
      losses[k] = loss.item()
    out[split] = losses.mean()
  model.train()
  return out

In [ ]:
class Head(nn.Module):

  def __init__(self,head_size):
    super().__init__()
    self.key = nn.Linear(n_embd, head_size, bias=False)
    self.query = nn.Linear(n_embd, head_size, bias = False)
    self.value = nn.Linear(n_embd, head_size, bias = False)
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
    self.dropout = nn.Dropout(dropout)

  def forward(self,x):
    B,T,C = x.shape
    k = self.key(x)
    q = self.query(x)
    v = self.value(x)

    wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5
    wei = wei.masked_fill(self.tril[:T,:T] == 0, float('-inf'))
    wei = F.softmax(wei, dim=-1)
    wei = self.dropout(wei)

    out = wei @ v
    return out


In [ ]:
class MultiHeadAttention(nn.Module):

  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(head_size * num_heads, n_embd)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads],dim=-1)
    out = self.dropout(self.proj(out))
    return out

In [ ]:
class FeedForward(nn.Module):

  def __init__(self,n_embd):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd, 4* n_embd),
        nn.ReLU(),
        nn.Linear(4*n_embd, n_embd),
        nn.Dropout(dropout),
    )

  def forward(self,x):
    return self.net(x)

In [ ]:
class Block(nn.Module):
  def __init__(self,n_embd,n_head):

    super().__init__()
    head_size = n_embd // n_head
    self.sa = MultiHeadAttention(n_head, head_size)
    self.ffwd = FeedForward(n_embd)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self,x):
    y = self.sa(x)
    x = self.ln1(x+y)
    y = self.ffwd(x)
    x = self.ln2(x+y)
    return x








In [ ]:
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head=n_head) for _ in range(n_layer)]
        )

        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def _init_weights(self, module):
      if isinstance(module, nn.Linear):
        torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if module.bias is not None:
          torch.nn.init.zeros_(module.bias)
      elif isinstance(module, nn.Embedding):
        torch.nn.init.normal_(module.weight, mean=0.0,std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # token + positional embeddings
        tok_emb = self.token_embedding_table(idx)              # (B, T, n_embd)
        pos_emb = self.position_embedding_table(
            torch.arange(T, device=idx.device)
        )                                                       # (T, n_embd)

        x = tok_emb + pos_emb                                  # (B, T, n_embd)

        # transformer blocks
        x = self.blocks(x)
        x = self.ln_f(x)

        # language model head
        logits = self.lm_head(x)                               # (B, T, vocab_size)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):

      for _ in range(max_new_tokens):

        idx_cond = idx[:, -block_size:]

        logits, _ = self.forward(idx_cond)

        logits = logits[:, -1, :]

        probs = F.softmax(logits, dim=-1)

        idx_next = torch.multinomial(probs, num_samples=1)

        idx = torch.cat((idx, idx_next), dim=1)
      return idx







In [ ]:
# Create a PyTorch optimizer
model = GPTLanguageModel(vocab_size=80).to(device=device)

optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

for iter in range(max_iters):
  if iter % eval_iters == 0:
    losses = estimate_loss()
    print(f'step: {iter}, loss{losses}')

  # sample a batch of data

  xb,yb = get_batch('train')

  logits, loss = model.forward(xb,yb)

  optimizer.zero_grad(set_to_none= True)

  loss.backward()
  optimizer.step()



step: 0, loss{'train': tensor(4.4950), 'val': tensor(4.4975)}
step: 100, loss{'train': tensor(2.3054), 'val': tensor(2.3752)}
step: 200, loss{'train': tensor(1.9274), 'val': tensor(2.0355)}
step: 300, loss{'train': tensor(1.6671), 'val': tensor(1.8158)}
step: 400, loss{'train': tensor(1.4954), 'val': tensor(1.6828)}
step: 500, loss{'train': tensor(1.3851), 'val': tensor(1.6179)}
step: 600, loss{'train': tensor(1.2965), 'val': tensor(1.5727)}
step: 700, loss{'train': tensor(1.2201), 'val': tensor(1.5421)}
step: 800, loss{'train': tensor(1.1690), 'val': tensor(1.5108)}
step: 900, loss{'train': tensor(1.1094), 'val': tensor(1.5099)}
step: 1000, loss{'train': tensor(1.0663), 'val': tensor(1.5020)}
step: 1100, loss{'train': tensor(1.0111), 'val': tensor(1.4957)}
step: 1200, loss{'train': tensor(0.9623), 'val': tensor(1.5216)}
step: 1300, loss{'train': tensor(0.9194), 'val': tensor(1.5280)}
step: 1400, loss{'train': tensor(0.8721), 'val': tensor(1.5481)}
step: 1500, loss{'train': tensor(0.82

In [ ]:
context = torch.zeros((1,1), dtype = torch.long, device=device)
generated_chars = decode(model.generate(context, max_new_tokens = 10000)[0].tolist())
print(generated_chars)



He took off the roof in which they saw nearly filled by at the succession of
priposite the pation.

"We descratch my friends, than it's looks at the poork of mother banks
her looked through surprise it and just above the Cloud Fairien]

"Could the Chief 3? And you the piglets in my appeared through the eager
cave, and thought make a might that had been ever breakful any disable to the
edge where Jim and clever heard to them such lived in the Presign of other
box for she his ead to be far out his lit heels, and forbing little with her face and so where are no pather
from it. So Dorothy had lost over our world agains, which wand sto
dart the out of the hall.

"Come here," answered the man's voice answered, please. "I'll need
yours, my dear, and if I had to do me with no ine, I'd the held over them to
Look and one and helped in a
lith thould them and they mice agreen speeche food in our
bodies.

"Can't worry, my dear," Dorothy exclaimed, drawing the boy, the harness too,
presently. "Fi!"